# PathSnap — Notebook 05: model selection (Milestone 5)
**Owner: B (GeniePlunder) with C (nisarg-ravi, who tunes the chosen model in M6).**

M4 showed three models tied on mean CV Top-1 (hist_gb 0.8520, random_forest 0.8512, mlp 0.8498; fold std ≈ 0.019).
This notebook decides **with evidence**, using only the 80 TRAINING trips (out-of-fold predictions from M4's
trip-grouped CV). Student-test numbers are shown for reference only and are NOT used to decide.

**Decision rule (declared before computing, decision D19):**
1. Primary metric: point-level Top-1 (OOF). Find every model that is **not significantly worse** than the best:
   the 95% trip-bootstrap CI of (best − model) includes 0.
2. Among those tied models, compare five secondary criteria that matter for the rest of the pipeline:
   (a) OOF log loss: probabilities become HMM emissions (M7); (b) calibration error (ECE); (c) stability: fold std
   of Top-1; (d) inference time per 1000 GPS points (the API on Laptop 1); (e) tuning cost: seconds per 5-fold CV
   evaluation (M6 Bayesian optimisation needs many). The model winning the most criteria is selected
   (ties → better OOF log loss).

In [ ]:
import os, sys, json, glob, time, pickle, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    M4, DS_DIR, OUT = f"{DRIVE}/m4_outputs", f"{DRIVE}/m3_outputs/dataset", f"{DRIVE}/m5_outputs"
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO): os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO if os.path.isdir(f"{REPO}/pathsnap") else DRIVE
else:
    M4 = os.environ.get("PATHSNAP_M4", "m4_outputs"); DS_DIR = os.environ.get("PATHSNAP_DS", "m3_outputs/dataset")
    OUT = os.environ.get("PATHSNAP_M5", "m5_outputs"); PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
from pathsnap.modeling import load_dataset, point_predictions
from pathsnap.model_zoo import ZOO
from pathsnap.features import FEATURES
from pathsnap.config import SEED
os.makedirs(OUT, exist_ok=True)
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

F, P = load_dataset(DS_DIR)
TR = F[F.partition == "train"].reset_index(drop=True)
R = {json.load(open(f))["model"]: json.load(open(f)) for f in sorted(glob.glob(f"{M4}/*_results.json"))}
MODELS = [m for m in ["hist_gb", "random_forest", "mlp", "lightgbm", "logreg", "extra_trees", "knn", "gaussian_nb"]
          if m in R and os.path.isfile(f"{M4}/{m}_oof.csv.gz")]
print("models with OOF predictions:", MODELS)
OOF = {}
for m in MODELS:
    o = pd.read_csv(f"{M4}/{m}_oof.csv.gz")
    assert (o.trip_id.values == TR.trip_id.values).all() and (o.pt.values == TR.pt.values).all() and (o.arc.values == TR.arc.values).all()
    OOF[m] = o.oof_proba.values

## 1. Per-trip Top-1 and paired comparison (primary metric)
Unit of comparison = **trip** (points inside a trip are not independent). For each model we compute OOF Top-1
per trip, then compare models on the same 80 trips: Wilcoxon signed-rank test + 95% bootstrap CI (2,000 resamples
of trips) of the pooled Top-1 difference vs the best model.

In [ ]:
lab_pts = P[(P.partition == "train") & (P.label_status == 0)].groupby("trip_id").size()
per_trip = {}
for m in MODELS:
    pp = point_predictions(TR, OOF[m]); pp = pp[pp.label_status == 0]
    per_trip[m] = (pp.groupby("trip_id").pred_is_ref.sum().reindex(lab_pts.index, fill_value=0))
CORR = pd.DataFrame(per_trip)                         # correct points per trip
TOP1 = CORR.div(lab_pts, axis=0)                      # Top-1 per trip
pooled = CORR.sum() / lab_pts.sum()
best = pooled.idxmax()
rng = np.random.default_rng(SEED); trips = lab_pts.index.values
B = 2000; idx = rng.integers(0, len(trips), size=(B, len(trips)))
Cm = CORR.loc[trips].values; Nl = lab_pts.loc[trips].values
boot = (Cm[idx].sum(1) / Nl[idx].sum(1)[:, None])     # (B, models) pooled Top-1 per resample
bi = MODELS.index(best)
rows = []
for j, m in enumerate(MODELS):
    d = boot[:, bi] - boot[:, j]
    w = wilcoxon(TOP1[best], TOP1[m]).pvalue if m != best else np.nan
    rows.append({"model": m, "oof_top1_pooled": pooled[m], "diff_vs_best": pooled[best] - pooled[m],
                 "ci95_low": np.percentile(d, 2.5), "ci95_high": np.percentile(d, 97.5),
                 "wilcoxon_p_vs_best": w, "trips_where_better_than_best": int((TOP1[m] > TOP1[best]).sum()),
                 "tied_with_best": bool(m == best or np.percentile(d, 2.5) <= 0)})
SIG = pd.DataFrame(rows).set_index("model"); SIG.to_csv(f"{OUT}/01_paired_top1_vs_best.csv"); display(SIG.round(4))
TIED = SIG[SIG.tied_with_best].index.tolist()
print(f"Best pooled OOF Top-1: {best} ({pooled[best]:.4f}). Not significantly worse (CI of difference includes 0): {TIED}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
order = SIG.sort_values("oof_top1_pooled", ascending=False).index
ax[0].errorbar(range(len(order)), SIG.loc[order, "diff_vs_best"],
               yerr=[SIG.loc[order, "diff_vs_best"] - SIG.loc[order, "ci95_low"], SIG.loc[order, "ci95_high"] - SIG.loc[order, "diff_vs_best"]],
               fmt="o", capsize=4); ax[0].axhline(0, c="k", lw=.8)
ax[0].set_xticks(range(len(order))); ax[0].set_xticklabels(order, rotation=30, ha="right")
ax[0].set(ylabel=f"Top-1 deficit vs {best}", title="OOF Top-1 deficit vs best, 95% trip-bootstrap CI (0 inside = tied)")
ax[1].boxplot([TOP1[m] for m in order], showfliers=True)       # no labels= kwarg: renamed across matplotlib versions
ax[1].set_xticks(range(1, len(order) + 1)); ax[1].set_xticklabels(order, rotation=30, ha="right")
ax[1].set(ylabel="Top-1 per trip (OOF)", title="Per-trip Top-1 distribution (80 training trips)")
plt.tight_layout(); plt.savefig(f"{OUT}/01_paired_comparison.png", dpi=130); plt.show()

## 2. Probability quality: log loss, Brier score, calibration (ECE)
In M7 the selected model's probabilities are the HMM **emission** scores, so they must be well calibrated:
when the model says 0.8, the candidate should be right about 80% of the time.

In [ ]:
lab = TR.label.values >= 0; y = TR.label.values[lab]
def ece(y, p, bins=15):
    e = np.linspace(0, 1, bins + 1); k = np.clip(np.digitize(p, e) - 1, 0, bins - 1); tot = 0.0
    for b in range(bins):
        m = k == b
        if m.any(): tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return tot
cal_rows, curves = [], {}
for m in MODELS:
    p = np.clip(OOF[m][lab], 1e-7, 1 - 1e-7)
    e = np.linspace(0, 1, 16); k = np.clip(np.digitize(p, e) - 1, 0, 14)
    curves[m] = pd.DataFrame({"p": p, "y": y, "k": k}).groupby("k").agg(pm=("p", "mean"), ym=("y", "mean"), n=("y", "size"))
    cal_rows.append({"model": m, "oof_log_loss": -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)),
                     "oof_brier": np.mean((p - y) ** 2), "oof_ece": ece(y, p)})
CAL = pd.DataFrame(cal_rows).set_index("model"); CAL.to_csv(f"{OUT}/02_calibration.csv"); display(CAL.round(4))
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], "k--", lw=.8, label="perfect calibration")
for m in MODELS:
    c = curves[m]; ax.plot(c.pm, c.ym, "o-", ms=3, label=f"{m} (ECE {CAL.loc[m, 'oof_ece']:.3f})")
ax.set(xlabel="predicted probability", ylabel="observed share correct", title="Reliability diagram (OOF, 80 training trips)")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/02_reliability.png", dpi=130); plt.show()

## 3. Stability, cost, model size and tuning cost
The three tied candidates are re-fitted once on all training trips here only to measure **model size** and
**single-trip latency** (what the API does per request). These refits are not used for anything else.

In [ ]:
cost_rows = []
for m in MODELS:
    folds = pd.read_csv(f"{M4}/{m}_folds.csv")
    cost_rows.append({"model": m, "fold_std_top1": folds.val_top1_accuracy.std(), "worst_fold_top1": folds.val_top1_accuracy.min(),
                      "cv_eval_seconds": folds.fit_s.sum() + folds.predict_s.sum(),
                      "infer_ms_per_1000_pts": R[m]["inference_ms_per_1000_points"], "train_s_full": R[m]["train_time_s"]})
COST = pd.DataFrame(cost_rows).set_index("model")
fit_rows = TR[TR.label >= 0]
TE = F[F.partition == "student_test"]
one_trip = TE[TE.trip_id == TE.groupby("trip_id").size().sort_values().index[len(TE.trip_id.unique()) // 2]]   # median-size trip
for m in TIED:
    if m not in ZOO: continue
    mod = ZOO[m]["make"](); kw = {}
    mod.fit(fit_rows[FEATURES].values, fit_rows.label.values)
    COST.loc[m, "model_size_mb"] = len(pickle.dumps(mod)) / 1e6
    t0 = time.time()
    for _ in range(5): mod.predict_proba(one_trip[FEATURES].values)
    COST.loc[m, "median_trip_latency_ms"] = (time.time() - t0) / 5 * 1000
COST["trials_in_1h_of_tuning"] = (3600 / COST.cv_eval_seconds).round(0)
COST.to_csv(f"{OUT}/03_stability_cost.csv"); display(COST.round(4))
print("median-size student-test trip used for latency:", int(one_trip.trip_id.iloc[0]), f"({one_trip[['trip_id','pt']].drop_duplicates().shape[0]} GPS points, {len(one_trip)} candidate rows)")

## 4. Behaviour where it matters: moving vs stopped (OOF, training trips)

In [ ]:
ctx = TR.drop_duplicates(["trip_id", "pt"])[["trip_id", "pt", "heading_known"]]
mv = []
for m in MODELS:
    pp = point_predictions(TR, OOF[m]); pp = pp[pp.label_status == 0].merge(ctx, on=["trip_id", "pt"])
    mv.append({"model": m, "oof_top1_moving": pp[pp.heading_known == 1].pred_is_ref.mean(),
               "oof_top1_stopped": pp[pp.heading_known == 0].pred_is_ref.mean()})
MV = pd.DataFrame(mv).set_index("model"); MV.to_csv(f"{OUT}/04_moving_vs_stopped_oof.csv"); display(MV.round(4))

## 5. Decision

In [ ]:
D = SIG[["oof_top1_pooled", "diff_vs_best", "ci95_low", "ci95_high", "tied_with_best"]].join(CAL).join(COST).join(MV)
D["student_test_top1 (reference only)"] = [R[m]["student_test"]["top1_accuracy"] for m in D.index]
D.to_csv(f"{OUT}/05_decision_table.csv"); display(D.round(4))
crit = {"oof_log_loss": "min", "oof_ece": "min", "fold_std_top1": "min", "infer_ms_per_1000_pts": "min", "cv_eval_seconds": "min"}
T = D.loc[TIED]
wins = pd.Series(0, index=TIED, dtype=int); winners = {}
for c, how in crit.items():
    w = T[c].idxmin(); wins[w] += 1; winners[c] = w
print("Tied set:", TIED); print("Winner per secondary criterion:", winners); print("Wins:", wins.to_dict())
top = wins[wins == wins.max()].index.tolist()
SELECTED = top[0] if len(top) == 1 else T.loc[top, "oof_log_loss"].idxmin()
print(f"\nSELECTED MODEL: {SELECTED}")
json.dump({"selected": SELECTED, "tied_set": TIED, "criterion_winners": winners, "wins": wins.to_dict(),
           "best_primary": best, "decision_table": D.reset_index().to_dict(orient="records")},
          open(f"{OUT}/selection.json", "w"), indent=1, default=float)

## 6. Robustness of the decision: do tiny differences decide it?
A raw "who wins most criteria" count treats a 0.0175 vs 0.0194 fold std as a win. Check: count a model as winning a
criterion if it is within **5% (relative) of the best value**, i.e. near-ties are shared. Also report what each criterion
means in practice.

In [ ]:
TOL = 0.05
wins_tol = pd.Series(0, index=TIED, dtype=int); share = {}
for c in crit:
    bestv = T[c].min(); w = T.index[T[c] <= bestv * (1 + TOL)].tolist(); share[c] = w
    for m in w: wins_tol[m] += 1
print("Winners within 5% of best, per criterion:", share); print("Wins (tolerant):", wins_tol.to_dict())
top_t = wins_tol[wins_tol == wins_tol.max()].index.tolist()
SELECTED_TOL = top_t[0] if len(top_t) == 1 else T.loc[top_t, "oof_log_loss"].idxmin()
print("Selected under tolerant counting:", SELECTED_TOL, "| strict counting:", SELECTED,
      "→", "SAME decision" if SELECTED_TOL == SELECTED else "DIFFERENT — inspect")
print("\nPractical meaning (tied set):")
for m in TIED:
    r = T.loc[m]
    print(f"  {m:14s} log loss {r.oof_log_loss:.4f} | ECE {r.oof_ece:.4f} | fold std {r.fold_std_top1:.4f} | "
          f"one-trip latency {r.median_trip_latency_ms:.0f} ms | size {r.model_size_mb:.1f} MB | ~{r.trials_in_1h_of_tuning:.0f} tuning trials per hour")
sel = json.load(open(f"{OUT}/selection.json")); sel.update({"selected_tolerant": SELECTED_TOL, "tolerant_winners": share,
                                                              "tolerant_wins": wins_tol.to_dict()})
json.dump(sel, open(f"{OUT}/selection.json", "w"), indent=1, default=float)